# Sprint 4: Validación y Benchmarking Final

Este notebook ejecuta la validación final del proyecto:

1. **Tests estadísticos** de significancia
2. **Comparación** exhaustiva con baselines
3. **Interpretación de negocio** de los resultados
4. **Generación del reporte final**

## Objetivo
Demostrar que el Algoritmo Genético supera estadísticamente a los modelos tradicionales de atribución.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from omnievo import (
    DataGenerator,
    GeneticOptimizer,
    cross_validate,
    run_benchmark,
    generate_final_report,
    compare_baselines,
    plot_convergence,
    plot_weights,
    plot_comparison,
    plot_prediction_scatter,
)

plt.style.use('seaborn-v0_8-whitegrid')
print("Librerías cargadas")

## 1. Preparación de Datos

In [ ]:
# Dataset grande para validación robusta
generator = DataGenerator(n_users=2000, random_state=42)
df = generator.generate()
channels = generator.get_channel_names()

X = df[channels].values
y = df['LTV_real'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Dataset: {len(df)} usuarios")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Canales ({len(channels)}): {channels}")

## 2. Entrenamiento con Parámetros Óptimos

Usamos los mejores parámetros encontrados en Sprint 3.

In [ ]:
# Parámetros óptimos del Sprint 3
best_params = {
    "population_size": 50,
    "generations": 100,
    "cxpb": 0.6,
    "mutpb": 0.2,
    "elite_size": 2,
}

print("Entrenando modelo con parámetros óptimos...")
optimizer = GeneticOptimizer(**best_params, random_state=42, verbose=True)
result = optimizer.fit(X_train, y_train)

print(f"\nEntrenamiento completado")
print(f"Mejor fitness: {result.best_fitness:.4f}")

## 3. Validación Cruzada para Tests Estadísticos

In [ ]:
# Validación cruzada 5-fold
cv_results = cross_validate(
    X, y,
    k_folds=5,
    optimizer_params=best_params,
    verbose=True,
)

print(f"\nResultados CV:")
print(f"  RMSE: {cv_results['rmse_mean']:.4f} ± {cv_results['rmse_std']:.4f}")
print(f"  Pearson: {cv_results['pearson_mean']:.4f} ± {cv_results['pearson_std']:.4f}")

## 4. Benchmarking Completo

In [ ]:
# Ejecutar benchmark completo
benchmark_report = run_benchmark(
    X_train, y_train,
    X_test, y_test,
    ga_weights=result.best_weights,
    channel_names=channels,
    cv_results=cv_results,
    verbose=True,
)

## 5. Visualizaciones Finales

In [ ]:
# Gráfica de convergencia
plot_convergence(result);
plt.title('Convergencia del Algoritmo Genético', fontsize=14, fontweight='bold');
plt.show()

In [ ]:
# Pesos de atribución
plot_weights(result.best_weights, channels);
plt.show()

In [ ]:
# Comparación con baselines
plot_comparison(benchmark_report.comparison_df);
plt.show()

In [ ]:
# Scatter plot predicción vs realidad
from omnievo.fitness import predict_ltv

y_pred = predict_ltv(X_test, result.best_weights, scale_factor=y_test.max())
plot_prediction_scatter(y_test, y_pred);
plt.show()

## 6. Análisis de Resultados por Tipo de Canal

In [ ]:
# Agrupar pesos por tipo de canal
channel_types = generator.get_channel_types()
weights = result.best_weights

type_weights = {}
for ch_type, ch_list in channel_types.items():
    indices = [channels.index(ch) for ch in ch_list]
    type_weights[ch_type] = sum(weights[i] for i in indices)

print("Peso total por tipo de canal:")
for ch_type, w in sorted(type_weights.items(), key=lambda x: -x[1]):
    bar = "█" * int(w * 40)
    print(f"  {ch_type.upper():10s} {w*100:5.1f}% {bar}")

In [ ]:
# Gráfica de distribución por tipo
fig, ax = plt.subplots(figsize=(8, 6))

types = list(type_weights.keys())
weights_by_type = [type_weights[t] * 100 for t in types]
colors = ['#3498db', '#2ecc71', '#e74c3c']

ax.pie(weights_by_type, labels=[t.upper() for t in types], 
       autopct='%1.1f%%', colors=colors, startangle=90,
       explode=[0.05 if w == max(weights_by_type) else 0 for w in weights_by_type])
ax.set_title('Distribución de Atribución por Tipo de Canal', fontsize=14, fontweight='bold')
plt.show()

## 7. Reporte Final

In [ ]:
# Generar reporte en markdown
final_report = generate_final_report(benchmark_report, output_format="markdown")
print(final_report)

In [ ]:
# Guardar reporte
with open('../results/final_report.md', 'w') as f:
    f.write(final_report)
print("Reporte guardado en results/final_report.md")

## 8. Conclusiones del Proyecto

### Hipótesis

**H₀ (Nula):** Los pesos del AG no difieren significativamente de una distribución uniforme.

**H₁ (Alternativa):** El AG produce un RMSE significativamente menor que los baselines.

In [ ]:
print("=" * 70)
print("CONCLUSIÓN DEL PROYECTO")
print("=" * 70)

if benchmark_report.hypothesis_rejected:
    print("\n✅ HIPÓTESIS NULA RECHAZADA")
    print("\nEl Algoritmo Genético demostró ser estadísticamente superior")
    print("a los modelos de atribución tradicionales.")
else:
    print("\n⚠️ HIPÓTESIS NULA NO RECHAZADA")
    print("\nNo se alcanzó significancia estadística, aunque")
    print("las métricas muestran mejora.")

print(f"\nMétricas Finales:")
print(f"  • RMSE: {benchmark_report.ga_rmse:.4f}")
print(f"  • Mejora vs Uniforme: {benchmark_report.improvement_pct:.1f}%")
print(f"  • Correlación Pearson: {benchmark_report.ga_pearson:.4f}")

print(f"\nTop 3 Canales Más Predictivos:")
for insight in benchmark_report.business_insights[:3]:
    print(f"  {insight.rank}. {insight.channel}: {insight.weight*100:.1f}%")

print("\n" + "=" * 70)
print("PROYECTO COMPLETADO ✓")
print("=" * 70)